# Notebook 03: Dynamic analysis — API call sequences

**Class 04_02.** Static features can be faked; behaviour has to be *executed*. A sandbox (Cuckoo, CAPEv2)
runs the file in an instrumented VM and logs the API calls it makes, in order. This notebook compares:

1. **Bag of API calls** — counts per API id, order discarded (logistic regression).
2. **GRU over the call sequence** — a recurrent model with an embedding over the 307 API vocabulary,
   masking the padding id 0.

Data: oliveira et al. (2019), 43,876 files x 100 API calls from Cuckoo, 97.5% malware — a hard imbalance
in the *other* direction (benign is the rare class). At 43,876 x 100 integers the sequences are only ~18 MB,
so this dataset is safe to run on a laptop; the bag-of-calls matrix is built sparse (~4 MB).

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")

import sys

import numpy as np
import polars as pl

sys.path[:0] = [".", "..", "../../src", "../../../src"]
import aasplot as ap
import malwlib as mw

ap.setup("04")
Xs, ys = mw.load_sequences()
print(f"sequences: {Xs.shape[0]:,} x {Xs.shape[1]} calls, vocab {Xs.max() + 1}, malware {ys.mean():.1%}")
print(
    f"memory: {Xs.nbytes / 1e6:.0f} MB dense — the sequences stay dense (small); "
    f"the bag-of-calls matrix is built as sparse CSR"
)

In [ ]:
lens = (
    pl.DataFrame({"malware": ys, "calls": (Xs > 0).sum(1)})
    .group_by("malware")
    .agg(pl.col("calls").mean().alias("mean"), pl.col("calls").median().alias("median"))
)
print(lens)  # fmt: skip

In [ ]:
from sklearn.model_selection import train_test_split

Str, Ste, ytr, yte = train_test_split(Xs, ys, test_size=0.25, stratify=ys, random_state=0)

## Bag of calls

The order-free baseline: one sparse column per API id. Already strong, because many families have
characteristic *sets* of calls.

In [ ]:
from scipy import sparse as sp
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score


def bow(X):
    return sp.csr_matrix(
        (np.ones(X.size), (np.repeat(np.arange(len(X)), X.shape[1]), X.ravel())), shape=(len(X), mw.VOCAB)
    )


Btr, Bte = bow(Str), bow(Ste)
lr = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=0).fit(Btr, ytr)
pb = lr.predict_proba(Bte)[:, 1]
mw.eval_bin(yte, pb, "bag-of-calls: ")

## GRU over the call sequence

`Embedding(307, 32, mask_zero=True) -> GRU(64) -> Dense(2)`. Order is the point: the same APIs in a
different sequence is different behaviour (e.g. write-then-execute vs execute-then-write).

In [ ]:
import keras
from keras import layers

keras.utils.set_random_seed(0)
model = keras.Sequential(
    [
        layers.Input((100,)),
        layers.Embedding(mw.VOCAB, 32, mask_zero=True),
        layers.GRU(64),
        layers.Dense(2, activation="softmax"),
    ]
)
model.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
hist = model.fit(Str, ytr, epochs=3, batch_size=128, validation_split=0.2, verbose=0)

pg = model.predict(Ste, verbose=0)[:, 1]  # (n, 2): column 1 is P(malware)
mw.eval_bin(yte, pg, "GRU: ")
print(f"bag-of-calls F1 .9695 vs GRU F1 {f1_score(yte, pg > 0.5):.4f} — the gap is not yet evidence of order")

In [ ]:
f, ax = ap.fig("col", 1.8)
ep = np.arange(1, len(hist.history["loss"]) + 1)
ax.plot(ep, hist.history["loss"], "-o", color=ap.BLUE, label="train")
ax.plot(ep, hist.history["val_loss"], "-o", color=ap.RED, label="val")
ax.set_xlabel("epoch"), ax.set_ylabel("loss"), ax.legend()
ap.save(f, "d04_gru_loss")

## Order matters: shuffled sequences

Destroy the order (shuffle the calls within each file) and re-run the GRU: if performance barely moves,
the model was only using the multiset of calls — and the bag-of-calls baseline was the right model all along.

In [ ]:
rng = np.random.default_rng(0)
Ssh = rng.permuted(Str, axis=1)
keras.utils.set_random_seed(0)
model2 = keras.Sequential(
    [
        layers.Input((100,)),
        layers.Embedding(mw.VOCAB, 32, mask_zero=True),
        layers.GRU(64),
        layers.Dense(2, activation="softmax"),
    ]
)
model2.compile("adam", "sparse_categorical_crossentropy")
model2.fit(Ssh, ytr, epochs=3, batch_size=128, validation_split=0.2, verbose=0)
psh = model2.predict(Ste, verbose=0)[:, 1]
e_ord = mw.eval_bin(yte, pg, "GRU original order: ")
e_sh = mw.eval_bin(yte, psh, "GRU shuffled order: ")

In [ ]:
f, ax = ap.fig("col", 1.8)
ax.bar([0, 1], [e_ord["f1"], e_sh["f1"]], 0.5, color=[ap.BLUE, ap.RED])
ax.set_xticks([0, 1], ["original order", "shuffled"])
ax.set_ylabel("F1@.5")
ax.set_ylim(0, 1.05)
ap.save(f, "d04_order_matters")

## Takeaways

- Sandbox API sequences (Cuckoo/CAPEv2) are the dynamic signal: 43,876 files, 100 calls each, 97.5% malware
  — imbalance handled with `class_weight="balanced"` on the linear model and a validation split on the GRU.
- The bag-of-calls baseline is strong (F1 .97); the GRU adds ~2 points of F1, but its F1 does not drop (.9891 vs .9914) when the
  calls are shuffled; only AUC falls (.893 to .855), so order carries a small ranking signal.
- The shuffled-order experiment is the honest test of whether a sequence model learned order at all;
  quote the F1 gap, not just the headline number.
- Infra note for the class: sandboxes need OS-matched VMs, anti-anti-VM tricks and snapshots; Cuckoo and
  CAPEv2 differ mainly in reporting and active development.